In [1]:
import pandas as pd
import numpy as np
import random
from mlxtend.frequent_patterns import apriori, association_rules
from surprise import Dataset, Reader, KNNBasic
from surprise.model_selection import train_test_split
from collections import defaultdict
import warnings 

warnings.filterwarnings("ignore") 

In [22]:
df = pd.read_csv('data/Faceplate.csv')
df.head(10)

,Transaction,Red,White,Blue,Orange,Green,Yellow
0,1,1,1,0,0,1,0
1,2,0,1,0,1,0,0
2,3,0,1,1,0,0,0
3,4,1,1,0,1,0,0
4,5,1,0,1,0,0,0
5,6,0,1,1,0,0,0
6,7,1,0,1,0,0,0
7,8,1,1,1,0,1,0
8,9,1,1,1,0,0,0
9,10,0,0,0,0,0,1


In [3]:
red_white = df[(df['Red'] == 1) & (df['White'] == 1)]
red_white_support = len(red_white)/len(df) * 100
print(f'Support of the itemset {{Red, White}}: {red_white_support:.2f}%')

Support of the itemset {Red, White}: 40.00%


In [4]:
# Functions
def get_sorted_association_rules(frequent_itemsets, columns=None, size=None, metric="confidence", min_threshold=0.5, sort_by='lift'):
    rules = association_rules(frequent_itemsets, metric=metric, min_threshold=min_threshold)
    if columns:
        rules = rules[columns]
    rules = rules.head(size)
    return rules.sort_values(by=sort_by, ascending=False)

In [5]:
itemsets = df[df.columns[1:]]  # Exclude the 'Transaction' column
frequent_itemsets = apriori(itemsets, min_support=0.2, use_colnames=True)
print(frequent_itemsets)

    support             itemsets
0       0.6                (Red)
1       0.7              (White)
2       0.6               (Blue)
3       0.2             (Orange)
4       0.2              (Green)
5       0.4         (White, Red)
6       0.4          (Blue, Red)
7       0.2         (Green, Red)
8       0.4        (White, Blue)
9       0.2      (White, Orange)
10      0.2       (Green, White)
11      0.2   (White, Blue, Red)
12      0.2  (Green, White, Red)


In [6]:
faceplate_params = {"frequent_itemsets":frequent_itemsets, "columns":['antecedents', 'consequents', 'support', 'confidence', 'lift', 'leverage']}
faceplate_rules = get_sorted_association_rules(**faceplate_params)
print(faceplate_rules)

       antecedents   consequents  support  confidence      lift  leverage
15         (Green)  (White, Red)      0.2    1.000000  2.500000      0.12
14    (White, Red)       (Green)      0.2    0.500000  2.500000      0.12
4          (Green)         (Red)      0.2    1.000000  1.666667      0.08
12  (Green, White)         (Red)      0.2    1.000000  1.666667      0.08
8          (Green)       (White)      0.2    1.000000  1.428571      0.06
7         (Orange)       (White)      0.2    1.000000  1.428571      0.06
13    (Green, Red)       (White)      0.2    1.000000  1.428571      0.06
3            (Red)        (Blue)      0.4    0.666667  1.111111      0.04
2           (Blue)         (Red)      0.4    0.666667  1.111111      0.04
5          (White)        (Blue)      0.4    0.571429  0.952381     -0.02
1            (Red)       (White)      0.4    0.666667  0.952381     -0.02
0          (White)         (Red)      0.4    0.571429  0.952381     -0.02
6           (Blue)       (White)      

_If Red and White faceplates are purchased, then with confidence of 50% Green faceplates will also be purchased. This rule has a lift ratio of 2.5_


In [7]:
top_six_rules = faceplate_rules.head(6)
print(top_six_rules)

       antecedents   consequents  support  confidence      lift  leverage
15         (Green)  (White, Red)      0.2         1.0  2.500000      0.12
14    (White, Red)       (Green)      0.2         0.5  2.500000      0.12
4          (Green)         (Red)      0.2         1.0  1.666667      0.08
12  (Green, White)         (Red)      0.2         1.0  1.666667      0.08
8          (Green)       (White)      0.2         1.0  1.428571      0.06
7         (Orange)       (White)      0.2         1.0  1.428571      0.06


In [24]:
charles_book_club_df = pd.read_csv('data/CharlesBookClub.csv') 

In [9]:
charles_book_club_df = charles_book_club_df.drop(columns=['Seq#', 'ID#', 'Gender','M', 'R', 'F', 'FirstPurch', 'Related Purchase', 'Mcode', 'Rcode', 'Fcode', 'Yes_Florence', 'No_Florence'])
charles_book_club_df = charles_book_club_df.map(lambda x: 1 if x > 0 else 0)
charles_book_club_df.head(10)

,ChildBks,YouthBks,CookBks,DoItYBks,RefBks,ArtBks,GeogBks,ItalCook,ItalAtlas,ItalArt,Florence
0,0,1,1,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,0
2,1,1,1,0,1,0,1,1,0,0,0
3,0,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,0
5,0,0,0,0,0,0,0,0,0,0,0
6,0,0,0,0,0,0,1,0,0,0,0
7,1,0,0,0,0,0,0,0,0,0,0
8,0,0,0,0,0,0,0,0,0,0,0
9,0,0,1,0,0,0,0,0,0,0,0


In [10]:
frequent_book_purchases = apriori(charles_book_club_df, min_support=0.05, use_colnames=True)
num_of_frequent_book_purchases = frequent_book_purchases.shape[0]
print("Number of frequent itemsets: ", num_of_frequent_book_purchases)

Number of frequent itemsets:  61


In [11]:
book_purchase_paramas = {"frequent_itemsets": frequent_book_purchases, "columns":['antecedents', 'consequents', 'support', 'confidence', 'lift', 'leverage'], "size": 25}
charles_book_purchase_rules = get_sorted_association_rules(**book_purchase_paramas)
print(charles_book_purchase_rules)

             antecedents          consequents  support  confidence      lift  \
21   (GeogBks, ChildBks)           (YouthBks)  0.07550    0.516239  2.166797   
17  (YouthBks, ChildBks)           (DoItYBks)  0.08025    0.544068  2.135693   
15            (YouthBks)  (CookBks, ChildBks)  0.12000    0.503673  2.081292   
24   (CookBks, ChildBks)           (DoItYBks)  0.12775    0.527893  2.072198   
16  (DoItYBks, YouthBks)           (ChildBks)  0.08025    0.777240  1.972690   
14  (YouthBks, ChildBks)            (CookBks)  0.12000    0.813559  1.958025   
18    (YouthBks, RefBks)           (ChildBks)  0.06200    0.763077  1.936743   
22   (DoItYBks, CookBks)           (ChildBks)  0.12775    0.757037  1.921414   
20  (YouthBks, ChildBks)            (GeogBks)  0.07550    0.511864  1.918892   
23  (DoItYBks, ChildBks)            (CookBks)  0.12775    0.791022  1.903783   
13   (YouthBks, CookBks)           (ChildBks)  0.12000    0.745342  1.891730   
19   (YouthBks, GeogBks)           (Chil

In [12]:
highest_support_rule = charles_book_purchase_rules.sort_values(by='support', ascending=False).iloc[0]
print(highest_support_rule)

antecedents    (ChildBks)
consequents     (CookBks)
support             0.242
confidence       0.614213
lift             1.478251
leverage         0.078293
Name: 2, dtype: object


In [13]:
highest_lift_rule = charles_book_purchase_rules.iloc[0]
print(highest_lift_rule)

antecedents    (GeogBks, ChildBks)
consequents             (YouthBks)
support                     0.0755
confidence                0.516239
lift                      2.166797
leverage                  0.040656
Name: 21, dtype: object


_The rule with the highest support has a support of 0.242 and a lift of only 1.478251. While the rule with the highest lift has a lift of 2.166797 and a support of only 0.0755, this means that even though the high support rule represent patterns that are significant from a business perspective, the patterns are mostly due to random chance rather than a specific relationship, while high lift rule represent rare but strong associations. Inspite this, due to its low support, often it is pruned by the Apriori algorithm if the minimum support threshold is set too high._


In [14]:
top_ten_lift_rules = charles_book_purchase_rules.iloc[0:10]
lowest_conf_rule = top_ten_lift_rules.sort_values(by='confidence', ascending=True).iloc[0]
print(f'Top 10 rules sorted by lift are:\n{top_ten_lift_rules}\n\n')
print("Among the 10, the rule with lowest confidence value is:\n")
print(lowest_conf_rule)

Top 10 rules sorted by lift are:
             antecedents          consequents  support  confidence      lift  \
21   (GeogBks, ChildBks)           (YouthBks)  0.07550    0.516239  2.166797   
17  (YouthBks, ChildBks)           (DoItYBks)  0.08025    0.544068  2.135693   
15            (YouthBks)  (CookBks, ChildBks)  0.12000    0.503673  2.081292   
24   (CookBks, ChildBks)           (DoItYBks)  0.12775    0.527893  2.072198   
16  (DoItYBks, YouthBks)           (ChildBks)  0.08025    0.777240  1.972690   
14  (YouthBks, ChildBks)            (CookBks)  0.12000    0.813559  1.958025   
18    (YouthBks, RefBks)           (ChildBks)  0.06200    0.763077  1.936743   
22   (DoItYBks, CookBks)           (ChildBks)  0.12775    0.757037  1.921414   
20  (YouthBks, ChildBks)            (GeogBks)  0.07550    0.511864  1.918892   
23  (DoItYBks, ChildBks)            (CookBks)  0.12775    0.791022  1.903783   

    leverage  
21  0.040656  
17  0.042674  
15  0.062344  
24  0.066101  
16  0.03957

In [15]:
random.seed(0)

num_transactions = 50
num_items = 9

# Create binary incidence matrix with randomly assigned items
data = np.zeros((num_transactions, num_items), dtype=int)
for i in range(num_transactions):
    num_selected = random.randint(1, num_items)
    selected_items = random.sample(range(num_items), num_selected)
    for j in selected_items:
        data[i][j] = 1

item_columns = [f'Item{k+1}' for k in range(num_items)]
transactions = pd.DataFrame(data, columns=item_columns)
print(transactions)

    Item1  Item2  Item3  Item4  Item5  Item6  Item7  Item8  Item9
0       1      1      1      1      1      1      1      0      0
1       1      1      1      1      1      1      0      1      1
2       1      1      1      1      1      1      1      1      1
3       1      0      1      1      1      1      0      1      1
4       1      1      1      1      1      1      1      1      1
5       0      0      0      1      0      0      0      0      1
6       1      0      1      0      0      0      1      1      0
7       1      0      1      0      1      1      0      1      1
8       0      0      0      0      0      1      0      0      1
9       1      1      1      1      1      1      1      1      1
10      1      0      1      1      0      0      0      0      0
11      1      1      0      0      0      1      0      1      1
12      1      1      0      0      0      1      0      0      0
13      1      1      1      1      1      1      1      1      1
14      1 

In [16]:
frequent_transactions = apriori(transactions, min_support=0.04, use_colnames=True)
transactions_params = {"frequent_itemsets": frequent_transactions, "min_threshold": 0.7, "columns":['antecedents', 'consequents', 'support', 'confidence', 'lift'], "size": 6}
transactions_association_rules = get_sorted_association_rules(**transactions_params)
print(transactions_association_rules)

  antecedents consequents  support  confidence      lift
0     (Item2)     (Item1)     0.42    0.807692  1.187783
2     (Item3)     (Item1)     0.50    0.806452  1.185958
1     (Item1)     (Item3)     0.50    0.735294  1.185958
5     (Item5)     (Item1)     0.44    0.785714  1.155462
3     (Item1)     (Item4)     0.50    0.735294  1.114082
4     (Item4)     (Item1)     0.50    0.757576  1.114082


_Although synthetic data was used, a rule with uplift of 1.187783 was obtained which is more or less significant_


In [17]:
random.seed(0)

ratings_data = {
    'userID': [random.randint(0, 999) for _ in range(5000)],
    'itemID': [random.randint(0, 99) for _ in range(5000)],
    'rating': [random.randint(1, 5) for _ in range(5000)]
}

ratings_df = pd.DataFrame(ratings_data)
ratings_df.head(10)

,userID,itemID,rating
0,864,49,1
1,394,90,1
2,776,32,2
3,911,25,3
4,430,62,2
5,41,60,1
6,265,33,5
7,988,65,2
8,523,51,2
9,497,16,5


In [18]:
reader = Reader()
dataset = Dataset.load_from_df(ratings_df, reader)

trainset, testset = train_test_split(dataset, test_size=0.25)
print("Trainset Dimensions are:")
print(f"Train ratings: {trainset.n_ratings}")
print(f"Train users: {trainset.n_users}")
print(f"Train items: {trainset.n_items}\n\n")
print("Testset Dimensions are:")
print(f"Test ratings: {len(testset)}")


Trainset Dimensions are:
Train ratings: 3750
Train users: 969
Train items: 100


Testset Dimensions are:
Test ratings: 1250


In [19]:
# Cosine similarity between users
similarity_options = {
    "name": "cosine",
    "user_based": False,
}

algorithm = KNNBasic(sim_options=similarity_options)
algorithm.fit(trainset)
predictions = algorithm.test(testset)
print(predictions[:10])

Computing the cosine similarity matrix...
Done computing similarity matrix.
[Prediction(uid=212, iid=10, r_ui=3.0, est=np.float64(2.8365351623322055), details={'actual_k': 6, 'was_impossible': False}), Prediction(uid=656, iid=59, r_ui=5.0, est=np.float64(2.0), details={'actual_k': 4, 'was_impossible': False}), Prediction(uid=604, iid=13, r_ui=1.0, est=np.float64(4.3090169943749475), details={'actual_k': 3, 'was_impossible': False}), Prediction(uid=292, iid=28, r_ui=4.0, est=np.float64(4.446006854661491), details={'actual_k': 2, 'was_impossible': False}), Prediction(uid=322, iid=78, r_ui=5.0, est=np.float64(2.7727108687655693), details={'actual_k': 4, 'was_impossible': False}), Prediction(uid=764, iid=80, r_ui=4.0, est=np.float64(3.0545253957327523), details={'actual_k': 4, 'was_impossible': False}), Prediction(uid=875, iid=76, r_ui=1.0, est=np.float64(4.0), details={'actual_k': 2, 'was_impossible': False}), Prediction(uid=848, iid=47, r_ui=4.0, est=np.float64(2.0), details={'actual_k':

In [20]:
def get_highest_rated(items, n=1):
    """Return the top rated item(s) for a user from a set of items."""
    top_rated = defaultdict(list)
    for uid, iid, r_ui, _, _ in items:
        top_rated[uid].append((iid, r_ui))

    # Sort the items for each user and retrieve the highest rated
    for uid, user_ratings in top_rated.items():
        user_ratings.sort(key=lambda x: x[1], reverse=True)
        top_rated[uid] = user_ratings[:n]

    return top_rated

In [21]:
highest_rated = get_highest_rated(predictions)
recommended_items = {}

for  uid, user_ratings in highest_rated.items():
    # Retrieve the 5 most similar items (neighbors)
    neighbor_inner_ids = algorithm.get_neighbors(user_ratings[0][0], k=5)

    # Convert Inner IDs back to Raw IDs (original IDs)
    neighbor_raw_ids = [algorithm.trainset.to_raw_iid(inner_id) for inner_id in neighbor_inner_ids]
    recommended_items[uid] = neighbor_raw_ids
    print(f"Recommended items for user {uid} are: {neighbor_raw_ids}")

Recommended items for user 212 are: [29, 32, 54, 49, 59]
Recommended items for user 656 are: [43, 66, 54, 53, 67]
Recommended items for user 604 are: [29, 54, 59, 67, 46]
Recommended items for user 292 are: [68, 53, 49, 59, 18]
Recommended items for user 322 are: [29, 66, 32, 54, 53]
Recommended items for user 764 are: [43, 32, 53, 62, 46]
Recommended items for user 875 are: [29, 43, 66, 68, 54]
Recommended items for user 848 are: [29, 32, 62, 49, 59]
Recommended items for user 755 are: [32, 68, 62, 46, 18]
Recommended items for user 226 are: [54, 53, 49, 46, 87]
Recommended items for user 334 are: [66, 32, 53, 62, 49]
Recommended items for user 676 are: [29, 18, 48, 65, 24]
Recommended items for user 452 are: [29, 32, 62, 49, 59]
Recommended items for user 206 are: [68, 54, 62, 67, 18]
Recommended items for user 946 are: [43, 59, 67, 80, 37]
Recommended items for user 429 are: [29, 66, 67, 80, 46]
Recommended items for user 443 are: [29, 43, 66, 32, 59]
Recommended items for user 985 